In [ ]:
# -*- coding: utf-8 -*-
#"""Trend_RSEI_GAP_FILL.ipynb

#Automatically generated by Colab.

#Original file is located at
#    <colab-notebook-url-removed>
#"""

# ============================================================
# Colab: Mann–Kendall (PC1) por município + estatísticas (mean/median/std)
# Saída: projects/rnvuefsppgm/assets/Luana_RSEI_TRENDS
# ============================================================

# -*- coding: utf-8 -*-

import os
import sys
import copy
import geemap
from google.colab import auth
from IPython.display import display
import re

# Initialize a map object.
import ee
# Trigger the authentication flow.
ee.Authenticate()
# Initialize the library.
# ee.Initialize(project=os.environ.get('EE_PROJECT'))
ee.Initialize(project=os.environ.get('EE_PROJECT'))

import geemap.core as geemap

# Cloud authentication.
from google.colab import auth
auth.authenticate_user()



In [ ]:
# ============================================================
# 1) PARÂMETROS DO USUÁRIO
# ============================================================

# Coleção com a banda PC1 (RSEI / PCA) já calculada por município
asset_indices = 'projects/nexgenmap/MapBiomas2/LANDSAT/DEGRADACAO/mosaics-ndfia'

# FeatureCollection com os polígonos de municípios (campo 'NM_MUN')
asset_city    = 'projects/ee-rnvuefsppgm/assets/CAP_JOIN2'

# Coleção de saída (ImageCollection) com os resultados por município
asset_out_root = 'projects/rnvuefsppgm/assets/Luana_RSEI_TRENDS'

# Lista de municípios a processar
MUNICIPIOS = [
  'Porto Velho'#,
  #'Rio Branco','Manaus','Boa Vista','Belém', #'Salvador',
  #'Macapá','Palmas','São Luís','Teresina',
  #'Fortaleza','Natal','João Pessoa','Recife','Maceió','Aracaju','Belo Horizonte',
  #'Vitória','Rio de Janeiro','São Paulo','Curitiba','Florianópolis','Porto Alegre','Campo Grande',
  #'Cuiabá','Goiânia','Brasília'
]



In [ ]:
# ============================================================
# 2) UTILITÁRIOS
# ============================================================

def sanitize_id(name):
    """Substitui caracteres problemáticos por underline para usar em IDs de assets."""
    return re.sub(r'[^A-Za-z0-9_]', '_', str(name))


def ensure_image_collection(ic_asset_id):
    """Garante que exista uma ImageCollection no asset indicado."""
    try:
        info = ee.data.getAsset(ic_asset_id)
    except Exception:
        info = None

    if info is not None:
        asset_type = info.get('type')
        if asset_type == 'IMAGE_COLLECTION':
            print(f"[OK] ImageCollection já existe: {ic_asset_id}")
            return
        else:
            print(f"[AVISO] Asset existe mas não é ImageCollection: {ic_asset_id} (type={asset_type}).")
            return

    try:
        ee.data.createAsset({'type': ee.data.ASSET_TYPE_IMAGE_COLL}, ic_asset_id)
        print(f"[OK] ImageCollection criada: {ic_asset_id}")
    except Exception as e:
        print(f"[ERRO] Falha ao criar ImageCollection {ic_asset_id}: {e}")




In [ ]:
# ============================================================
# 3) GAP-FILL TEMPORAL (PC1)
# ============================================================

def temporal_gapfill_ic_bidirectional(ic, band_name='PC1'):
    """Preenche pixels mascarados usando vizinhos temporais (t+1 e t-1)."""
    ic = ic.select([band_name])

    ic_asc  = ic.sort('system:time_start')
    size    = ic_asc.size()
    ic_desc = ic_asc.sort('system:time_start', False)
    list_desc = ic_desc.toList(size)

    init_acc = ee.Dictionary({
        'prev': ee.Image(0).updateMask(ee.Image(0)),
        'list': ee.List([])
    })

    def _step_backward(el, acc):
        acc = ee.Dictionary(acc)
        prev = ee.Image(acc.get('prev'))
        img  = ee.Image(el)
        filled = img.unmask(prev)
        filled = filled.copyProperties(img, img.propertyNames())
        return ee.Dictionary({
            'prev': filled,
            'list': ee.List(acc.get('list')).add(filled)
        })

    back_dict = ee.Dictionary(list_desc.iterate(_step_backward, init_acc))
    list_back_asc = ee.List(back_dict.get('list')).reverse()
    ic_back_asc = ee.ImageCollection.fromImages(list_back_asc)

    list_fwd = ic_back_asc.toList(size)

    init_acc_fwd = ee.Dictionary({
        'prev': ee.Image(0).updateMask(ee.Image(0)),
        'list': ee.List([])
    })

    def _step_forward(el, acc):
        acc = ee.Dictionary(acc)
        prev = ee.Image(acc.get('prev'))
        img  = ee.Image(el)
        filled = img.unmask(prev)
        filled = filled.copyProperties(img, img.propertyNames())
        return ee.Dictionary({
            'prev': filled,
            'list': ee.List(acc.get('list')).add(filled)
        })

    fwd_dict = ee.Dictionary(list_fwd.iterate(_step_forward, init_acc_fwd))
    list_filled = ee.List(fwd_dict.get('list'))
    ic_filled = ee.ImageCollection.fromImages(list_filled)

    return ic_filled.sort('system:time_start').select([band_name])




In [ ]:
# ============================================================
# 4) NORMALIZAÇÃO 0–1 (para estatísticas de PC1)
# ============================================================

def normalize_ic_01(ic, band_name, region, scale=30):
    """Normaliza (0–1) uma ImageCollection usando min/max globais na região."""
    minImgT = ic.select(band_name).min()
    maxImgT = ic.select(band_name).max()

    minDict = minImgT.reduceRegion(
        reducer=ee.Reducer.min(),
        geometry=region,
        scale=scale,
        maxPixels=1e13,
        bestEffort=True
    )
    maxDict = maxImgT.reduceRegion(
        reducer=ee.Reducer.max(),
        geometry=region,
        scale=scale,
        maxPixels=1e13,
        bestEffort=True
    )

    minv = ee.Number(ee.Dictionary(minDict).get(band_name, 0))
    maxv = ee.Number(ee.Dictionary(maxDict).get(band_name, 1))
    rng  = maxv.subtract(minv)
    safe_rng = ee.Number(ee.Algorithms.If(rng.gt(0), rng, 1))

    minImg   = ee.Image.constant(minv)
    safeRImg = ee.Image.constant(safe_rng)

    def _map_norm(img):
        img = ee.Image(img)
        norm = img.select(band_name).subtract(minImg).divide(safeRImg).rename(band_name)
        norm = norm.copyProperties(img, ['system:time_start'])
        return norm

    ic_norm = ic.map(_map_norm)
    return {'collection': ic_norm, 'min': minv, 'max': maxv}




In [ ]:
# ============================================================
# 5) CÁLCULO DO MANN–KENDALL (seguindo a documentação oficial)
# ============================================================

def build_mk_image(municipio):
    """Constrói a imagem com bandas S, Z, significant, mean, median, stdDev para um município."""

    # Geometria
    fc_city = ee.FeatureCollection(asset_city).filter(ee.Filter.eq('NM_MUN', municipio))
    geom_city = fc_city.geometry()

    # Série PC1 por município
    ic_raw = (ee.ImageCollection(asset_indices)
              .filter(ee.Filter.eq('NM_MUN', municipio))
              .sort('system:time_start'))

    n_info = ic_raw.size().getInfo()
    if not n_info or n_info < 2:
        raise ValueError(f'Série com {n_info} imagem(ns); precisa de >= 2.')

    bands0 = ee.Image(ic_raw.first()).bandNames().getInfo()
    if 'PC1' not in bands0:
        raise ValueError("Banda 'PC1' não encontrada na coleção.")

    ic_pc1 = ic_raw.select(['PC1']).map(lambda img: ee.Image(img).clip(geom_city))

    # Gap-fill temporal
    ic_pc1_filled = temporal_gapfill_ic_bidirectional(ic_pc1, 'PC1')

    # -------- Coleção discreta (inteira) para o teste MK (como no tutorial) --------
    def to_discrete(img):
        img = ee.Image(img).select('PC1')
        scaled = img.multiply(10000).round().toInt16().rename('PC1')
        scaled = scaled.copyProperties(img, ['system:time_start'])
        return scaled

    coll_discrete = ic_pc1_filled.map(to_discrete)

    # -------- Coleção normalizada 0–1 para estatísticas --------
    norm_res = normalize_ic_01(ic_pc1_filled, 'PC1', geom_city, scale=30)
    coll_norm = ee.ImageCollection(norm_res['collection'])

    # -------- Join temporal para pares (i, j), i < j --------
    after_filter = ee.Filter.lessThan(
        leftField='system:time_start',
        rightField='system:time_start'
    )
    joined = ee.ImageCollection(
        ee.Join.saveAll(matchesKey='after').apply(
            primary=coll_discrete,
            secondary=coll_discrete,
            condition=after_filter
        )
    )

    # -------- Mann–Kendall S (soma dos sinais) --------
    def sign(i, j):
        # Função original do tutorial, agora aplicada em dados inteiros
        return (ee.Image(j)
                .neq(i)
                .multiply(ee.Image(j).subtract(i).clamp(-1, 1))
                .int())

    kendall = (ee.ImageCollection(
        joined.map(lambda current:
            ee.ImageCollection.fromImages(
                ee.Image(current).get('after')
            ).map(
                lambda image: ee.Image(sign(current, image)).unmask(0)
            )
        ).flatten()
    )
    .reduce(ee.Reducer.sum(), 2)
    .rename('S')
    .clip(geom_city))

    # -------- Variância do MK (com correção para empates) --------
    groups = coll_discrete.map(
        lambda i: ee.Image(i).multiply(
            coll_discrete.map(lambda j: ee.Image(i).eq(j)).sum().gt(1)
        )
    )

    def group(array):
        length = array.arrayLength(0)
        indices = (ee.Image([1])
                   .arrayRepeat(0, length)
                   .arrayAccum(0, ee.Reducer.sum())
                   .toArray(1))
        sorted_arr = array.arraySort()
        left = sorted_arr.arraySlice(0, 1)
        right = sorted_arr.arraySlice(0, 0, -1)
        mask = left.neq(right).arrayCat(ee.Image(ee.Array([[1]])), 0)
        run_indices = indices.arrayMask(mask)
        group_sizes = (run_indices.arraySlice(0, 1)
                       .subtract(run_indices.arraySlice(0, 0, -1)))
        return group_sizes

    def factors(image):
        return image.expression('b() * (b() - 1) * (b() * 2 + 5)')

    group_sizes = group(groups.toArray())
    group_factors = factors(group_sizes)
    group_factor_sum = group_factors.arrayReduce(
        ee.Reducer.sum(), [0]).arrayGet([0, 0])

    count = joined.count()
    kendall_variance = (factors(count)
                        .subtract(group_factor_sum)
                        .divide(18)
                        .float())

    # -------- Z, p e significância --------
    zero = kendall.multiply(kendall.eq(0))
    pos = kendall.multiply(kendall.gt(0)).subtract(1)
    neg = kendall.multiply(kendall.lt(0)).add(1)

    z = (zero
         .add(pos.divide(kendall_variance.sqrt()))
         .add(neg.divide(kendall_variance.sqrt()))
         .rename('Z')
         .clip(geom_city))

    def ee_cdf(z_img):
        return (ee.Image(0.5)
                .multiply(ee.Image(1).add(
                    ee.Image(z_img).divide(ee.Image(2).sqrt()).erf())))

    p = (ee.Image(1).subtract(ee_cdf(z.abs()))
         .rename('p')
         .clip(geom_city))

    significant = p.lte(0.025).rename('significant').toByte().clip(geom_city)

    # -------- Estatísticas temporais (PC1 normalizada 0–1) --------
    mean_img   = coll_norm.reduce(ee.Reducer.mean()).rename(['mean']).toFloat().clip(geom_city)
    median_img = coll_norm.reduce(ee.Reducer.median()).rename(['median']).toFloat().clip(geom_city)
    std_img    = coll_norm.reduce(ee.Reducer.stdDev()).rename(['stdDev']).toFloat().clip(geom_city)

    out_img = ee.Image.cat([
        significant.toFloat(),
        z.toFloat(),
        kendall.toFloat(),
        mean_img,
        median_img,
        std_img
    ]).rename(['significant', 'Z', 'S', 'mean', 'median', 'stdDev']).clip(geom_city)

    out_img = out_img.set({
        'NM_MUN': municipio,
        'mk:n_images': int(n_info),
        'mk:p_threshold': 0.025,
        'mk:description': 'Mann-Kendall (S, Z, significant) + estatísticas temporais (mean/median/stdDev) da PC1'
    })

    return out_img, geom_city




In [ ]:
# ============================================================
# 6) FUNÇÃO DE EXPORTAÇÃO POR MUNICÍPIO
# ============================================================

def mk_for_municipio(municipio):
    try:
        out_img, geom_city = build_mk_image(municipio)
    except ValueError as e:
        return {'municipio': municipio, 'status': 'pulado', 'msg': str(e)}
    except Exception as e:
        return {'municipio': municipio, 'status': 'erro', 'msg': str(e)}

    muni_safe = sanitize_id(municipio)
    asset_id_out = f"{asset_out_root}/MK_{muni_safe}"

    task = ee.batch.Export.image.toAsset(
        image=out_img,
        description=f"MK_{muni_safe}",
        assetId=asset_id_out,
        region=geom_city,
        scale=30,
        maxPixels=1e13
    )

    return {
        'municipio': municipio,
        'status': 'ok',
        'asset_out': asset_id_out,
        'task': task
    }




In [ ]:
# ============================================================
# 7) VISUALIZAÇÃO RÁPIDA (ON THE FLY) PARA TESTE (ex.: Salvador)
# ============================================================

def preview_mann_kendall(municipio):
    """Cria um mapa geemap com a banda S (MK) e os pixels significativos."""
    out_img, geom_city = build_mk_image(municipio)

    kendall_s = out_img.select('S')
    signif = out_img.select('significant')

    m = geemap.Map()
    m.centerObject(geom_city, 8)

    vis_s = {
        'min': -50,   # ajuste se necessário após inspecionar o histograma
        'max': 50,
        'palette': ['red', 'white', 'green']
    }
    m.addLayer(kendall_s, vis_s, f'MK S - {municipio}')

    vis_sig = {
        'min': 0,
        'max': 1,
        'palette': ['000000', 'ffffff']
    }
    m.addLayer(signif.updateMask(signif), vis_sig,
               f'Significant (p<=0.025) - {municipio}')

    return m




In [ ]:
# ============================================================
# 8) GARANTE A IMAGECOLLECTION DE SAÍDA E RODA PRÉVIA
# ============================================================

ensure_image_collection(asset_out_root)

# Pré-visualização apenas para o primeiro município
if len(MUNICIPIOS) > 0:
    preview_city = MUNICIPIOS[0]
    print(f"Gerando visualização de teste para {preview_city} (Mann-Kendall)...")
    preview_map = preview_mann_kendall(preview_city)
    display(preview_map)  # no Colab, isso abre o mapa interativo




In [ ]:
# ============================================================
# 9) LOOP PRINCIPAL – CRIA E INICIA AS TASKS DE EXPORTAÇÃO
# ============================================================

results = []
for mname in MUNICIPIOS:
    info = mk_for_municipio(mname)
    results.append(info)
    if info.get('status') == 'ok':
        print(f"[OK] Task criada: {info['municipio']} -> {info['asset_out']}")
        if 'task' in info and info['task'] is not None:
            info['task'].start()
        else:
            print(f"[ERRO] Task não encontrada no resultado para {info['municipio']}")
    else:
        print(f"[{info.get('status','erro').upper()}] {mname}: {info.get('msg','')}")

print("\nDica: monitore as tasks com:\nfor t in ee.batch.Task.list():\n    print(t.status())")
